# Advanced Graph RAG: Contextual Cypher Retrieval

You can improve the vector retriever by using a custom Cypher query to provide richer, more contextual answers.

You will need to:

- Create a Cypher `retrieval_query` that will be used with the retriever
- Use the `VectorCypherRetriever` class to create the retriever
- Create a `GraphRAG` pipeline that uses the retriever

**Prerequisites:** Complete [02 Embeddings](02_embeddings.ipynb) first to populate the graph with embeddings and create the vector index.

---

Import the required Python modules and set up the connection.

## Install Dependencies

First, install the required packages. This only needs to be run once per session.

In [ ]:
# Install neo4j-graphrag with Bedrock support
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" python-dotenv pydantic-settings nest-asyncio -q

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG

from data_utils import Neo4jConnection, get_llm, get_embedder

## Connect to Neo4j

Create and verify the connection to your Neo4j graph database.

In [ ]:
neo4j = Neo4jConnection().verify()
driver = neo4j.driver

## Initialize LLM and Embedder

Set up the Large Language Model (LLM) and the embedding model from AWS Bedrock. Both are configured in `CONFIG.txt` via `MODEL_ID` and `EMBEDDING_MODEL_ID`.

In [ ]:
# Initialize LLM and Embedder from AWS Bedrock
llm = get_llm()
embedder = get_embedder()

print(f"LLM: {llm.model_id}")
print(f"Embedder: {embedder.model_id}")

## VectorCypherRetriever with Custom Query

Create a `VectorCypherRetriever` that uses a Cypher query to return additional data from the graph.

In this example, we'll retrieve the chunk text along with information about adjacent chunks and the source document.

In [ ]:
# Custom Cypher query that returns chunk context with document info and adjacent chunks
context_query = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
RETURN 
    node.text AS context,
    doc.path AS document,
    node.index AS chunk_index,
    prev.text AS previous_chunk,
    next.text AS next_chunk
"""

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=context_query
)

print("VectorCypherRetriever initialized!")

**How it works:**  

- **Custom Cypher Query:**  
  The `context_query` matches text chunks (`node`) to their source documents and adjacent chunks.

  It returns:
  
  1. The chunk text as context
  2. The source document path
  3. The chunk index
  4. Previous and next chunk text for additional context

- **VectorCypherRetriever:**  
  - Performs semantic search using the `chunkEmbeddings` vector index.
  - Applies the Cypher `retrieval_query` to retrieve relevant context and related data.

---

Use the `GraphRAG` class to run a pipeline that uses the `vector_cypher_retriever`.

In [ ]:
# Initialize GraphRAG and Perform Search
query = "What products and services does Apple offer?"

rag = GraphRAG(llm=llm, retriever=vector_cypher_retriever)
response = rag.search(query, retriever_config={"top_k": 3}, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of results returned: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

## Viewing the Retrieved Context

You can inspect the context that was retrieved and passed to the LLM. This helps verify the relevance of the data.

In [ ]:
# View the context used in this query
print("Retrieved Context:")
print("=" * 60)
for i, item in enumerate(response.retriever_result.items):
    print(f"\n[Result {i+1}]")
    print(item.content)

## Expanded Context Window

One powerful use of VectorCypherRetriever is to expand the context window by including adjacent chunks. This provides the LLM with more surrounding context.

In [ ]:
# Query that combines current chunk with adjacent chunks for expanded context
expanded_context_query = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
WITH node, doc, prev, next
RETURN 
    COALESCE(prev.text + ' ', '') + node.text + COALESCE(' ' + next.text, '') AS expanded_context,
    doc.path AS source_document,
    node.index AS center_chunk_index
"""

expanded_retriever = VectorCypherRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    retrieval_query=expanded_context_query
)

# Test with expanded context
query = "Tell me about Apple's services"
rag_expanded = GraphRAG(llm=llm, retriever=expanded_retriever)
response = rag_expanded.search(query, retriever_config={"top_k": 2}, return_context=True)

print(f"Query: \"{query}\"\n")
print("Answer:")
print(response.answer)

In [ ]:
# View the expanded context
print("\nExpanded Context (includes adjacent chunks):")
print("=" * 60)
for i, item in enumerate(response.retriever_result.items):
    print(f"\n[Result {i+1}]")
    print(item.content)

## Comparing Standard vs Expanded Context

Let's compare the answers from the standard VectorRetriever with the expanded context VectorCypherRetriever.

In [ ]:
# Standard retriever (no graph traversal)
standard_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text']
)

query = "What does Apple do?"

# Standard retriever
print("=== Standard VectorRetriever ===")
rag_standard = GraphRAG(llm=llm, retriever=standard_retriever)
response_standard = rag_standard.search(query, retriever_config={"top_k": 2})
print(response_standard.answer)

# Expanded context retriever
print("\n=== VectorCypherRetriever (Expanded Context) ===")
response_expanded = rag_expanded.search(query, retriever_config={"top_k": 2})
print(response_expanded.answer)

## Summary

In this notebook, you learned:

1. **VectorCypherRetriever** - Combines vector search with custom Cypher queries
2. **Custom retrieval queries** - Traverse graph relationships to gather additional context
3. **Expanded context windows** - Include adjacent chunks for richer LLM context
4. **Graph-enhanced answers** - Leverage graph structure for more comprehensive responses

The VectorCypherRetriever pattern is particularly powerful when:
- You need to include related entities alongside retrieved text
- You want to expand the context window with adjacent chunks
- Your graph has rich relationships that provide valuable context

---

**Congratulations!** You've completed the Knowledge Graph lab series.

In [ ]:
# Cleanup
neo4j.close()